# Step 1~2 | 데이터 수집 & 가공
> **Team Sparta | 주식/코인 모니터링 웹앱 프로젝트**

---

## 이번 노트북에서 만들 것
1. `yfinance` / `pyupbit` API로 실제 데이터 받아오기
2. 종목 정보 조회 함수 만들기
3. 기간별 가격 히스토리 함수 만들기
4. 등락률 / 이동평균 계산 함수 만들기

> **빈칸(`___`)을 채우면서 진행하세요. 힌트는 주석을 확인하세요!**

## 환경 준비

In [3]:
import sys
!{sys.executable} -m pip install pyupbit
# 패키지 설치 (최초 1회만 실행)
#!pip install yfinance streamlit plotly pandas pyupbit

!{sys.executable} -m pip install yfinance


[notice] A new release of pip is available: 23.0.1 -> 26.1.1
[notice] To update, run: python3 -m pip install --upgrade pip
  Using cached yfinance-1.3.0-py2.py3-none-any.whl (133 kB)
  Using cached beautifulsoup4-4.14.3-py3-none-any.whl (107 kB)
  Using cached curl_cffi-0.15.0-cp310-abi3-macosx_10_9_x86_64.whl (2.8 MB)
  Using cached peewee-4.0.5-py3-none-any.whl (144 kB)
  Using cached protobuf-7.34.1-cp310-abi3-macosx_10_9_universal2.whl (429 kB)
  Using cached multitasking-0.0.13-py3-none-any.whl (16 kB)
  Using cached soupsieve-2.8.3-py3-none-any.whl (37 kB)
  Using cached rich-15.0.0-py3-none-any.whl (310 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 184.3/184.3 kB 5.3 MB/s eta 0:00:00
  Using cached pycparser-3.0-py3-none-any.whl (48 kB)
  Using cached markdown_it_py-4.2.0-py3-none-any.whl (91 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl (10.0 kB)
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider 

In [1]:
print('hello world')

hello world


In [ ]:
# 필요한 라이브러리를 import 하세요
import yfinance as yf       # 힌트: y로 시작하는 주식 라이브러리
import pandas as pd         # 힌트: 보통 pd로 줄여 씀
import numpy as np          # 힌트: 수치 계산 라이브러리
from datetime import datetime, timedelta

print('라이브러리 import 완료!')

라이브러리 import 완료!


---
##  yfinance API 탐색
본격적으로 함수를 만들기 전에, 데이터가 어떻게 생겼는지 먼저 확인합니다.

In [15]:
# 탐색용 코드 — 수정하지 말고 그대로 실행해서 구조를 확인하세요

ticker = yf.Ticker('AAPL')

# info: 종목의 다양한 정보가 담긴 딕셔너리
info = ticker.info
print('=== 사용 가능한 키 목록 (일부) ===')
keys_to_show = ['currentPrice', 'previousClose', 'shortName', 'currency', 'marketCap']
for key in keys_to_show:
    print(f'  {key}: {info.get(key)}')

=== 사용 가능한 키 목록 (일부) ===
  currentPrice: 293.257
  previousClose: 287.486
  shortName: Apple Inc.
  currency: USD
  marketCap: 4307169837056


In [16]:
#  히스토리 데이터 구조 확인
hist = ticker.history(period='1mo')  # 최근 1달

print('=== DataFrame 컬럼 ===')
print(hist.columns.tolist())
print()
print('=== 최근 5일 데이터 ===')
print(hist.tail())
print()
print(f'행 수: {len(hist)}, 컬럼 수: {len(hist.columns)}')

=== DataFrame 컬럼 ===
['Open', 'High', 'Low', 'Close', 'Volume', 'Dividends', 'Stock Splits']

=== 최근 5일 데이터 ===
                                 Open        High         Low       Close  \
Date                                                                        
2026-05-04 00:00:00-04:00  279.660004  280.630005  274.859985  276.829987   
2026-05-05 00:00:00-04:00  276.929993  284.570007  276.500000  284.179993   
2026-05-06 00:00:00-04:00  281.920013  288.029999  281.070007  287.510010   
2026-05-07 00:00:00-04:00  289.269989  292.130005  285.779999  287.440002   
2026-05-08 00:00:00-04:00  290.010010  294.760010  290.000000  293.320007   

                             Volume  Dividends  Stock Splits  
Date                                                          
2026-05-04 00:00:00-04:00  46668400        0.0           0.0  
2026-05-05 00:00:00-04:00  49311700        0.0           0.0  
2026-05-06 00:00:00-04:00  58336100        0.0           0.0  
2026-05-07 00:00:00-04:00  452243

In [17]:
# period 옵션 종류 확인
# period 값: '1d', '5d', '1mo', '3mo', '6mo', '1y', '2y', '5y'

for period in ['1wk', '1mo', '3mo', '1y']:
    df = ticker.history(period=period)
    print(f'period={period!r:8s} → {len(df):3d}행,  기간: {df.index[0].date()} ~ {df.index[-1].date()}')

period='1wk'    →   5행,  기간: 2026-05-04 ~ 2026-05-08
period='1mo'    →  22행,  기간: 2026-04-09 ~ 2026-05-08
period='3mo'    →  63행,  기간: 2026-02-09 ~ 2026-05-08
period='1y'     → 251행,  기간: 2025-05-09 ~ 2026-05-08


---
## 함수 1 | `get_stock_info()` — 종목 현재 정보 조회

### 요구사항
- 입력: 티커 심볼 (예: `'AAPL'`, `'005930.KS'`)
- 출력: 아래 키를 포함한 딕셔너리

```python
{
    'name': '애플',          # 종목명
    'current_price': 182.5,  # 현재가
    'prev_close': 180.0,     # 전일 종가
    'currency': 'USD',       # 통화
}
```
- 잘못된 티커나 오류 시 `None` 반환 + 오류 메시지 출력

In [ ]:
def get_stock_info(ticker_symbol):
    """
    주식 종목의 현재 정보를 조회합니다.
    
    Args:
        ticker_symbol (str): 티커 심볼 (예: 'AAPL', '005930.KS')
    
    Returns:
        dict: 종목 정보 딕셔너리 또는 오류 시 None
    """
    try:
        # 1단계: yf.Ticker 객체 생성
        # 힌트: yf.Ticker(___) 형태로 작성
        ticker = yf.Ticker(ticker_symbol)
        
        # 2단계: ticker.info 가져오기
        # 힌트: 위 탐색 코드에서 봤던 방법 그대로
        info = ticker.info
        #print('info === ',info)
        
        # 3단계: 필요한 값 추출 (없으면 기본값 반환)
        # 힌트: info.get('키이름', 기본값) 사용
        result = {
            'name':          info.get('shortName', ticker_symbol),      # 힌트: 'shortName'
            'current_price': info.get('currentPrice', 0),               # 힌트: 'currentPrice'
            'prev_close':    info.get('previousClose', 0),              # 힌트: 'previousClose'
            'currency':      info.get('currency', 'USD'),               # 힌트: 'currency'
        }
        
        # 4단계: current_price가 0이면 데이터가 없는 것 → None 반환
        if result['current_price'] == 0:
            print(f'[경고] {ticker_symbol}: 가격 데이터를 찾을 수 없습니다.')
            return None
        
        return result
    
    except Exception as e:
        # 힌트: f-string으로 ticker_symbol과 e를 포함한 오류 메시지 출력
        print(f'[오류] {ticker_symbol}: {e.with_traceback}')
        return None
    
    
print('get_stock_info => ',get_stock_info('AAPL'))

info ===  {'address1': 'One Apple Park Way', 'city': 'Cupertino', 'state': 'CA', 'zip': '95014', 'country': 'United States', 'phone': '(408) 996-1010', 'website': 'https://www.apple.com', 'industry': 'Consumer Electronics', 'industryKey': 'consumer-electronics', 'industryDisp': 'Consumer Electronics', 'sector': 'Technology', 'sectorKey': 'technology', 'sectorDisp': 'Technology', 'longBusinessSummary': 'Apple Inc. designs, manufactures, and markets smartphones, personal computers, tablets, wearables, and accessories worldwide. The company offers iPhone, a line of smartphones; Mac, a line of personal computers; iPad, a line of multi-purpose tablets; and wearables, home, and accessories comprising AirPods, Apple Vision Pro, Apple TV, Apple Watch, Beats products, and HomePod, as well as Apple branded and third-party accessories. It also provides AppleCare support and cloud services; and operates various platforms, including the App Store that allow customers to discover and download applic

In [12]:
# 함수 테스트
print('=== 정상 케이스 ===')
result = get_stock_info('AAPL')
print(result)

print()
print('=== 한국 주식 ===')
result_kr = get_stock_info('005930.KS')  # 삼성전자
print(result_kr)

print()
print('=== 오류 케이스 (잘못된 티커) ===')
result_err = get_stock_info('INVALID_TICKER_XYZ')
print(result_err)  # None이 출력되어야 함

=== 정상 케이스 ===
{'name': 'Apple Inc.', 'current_price': 293.257, 'prev_close': 287.486, 'currency': 'USD'}

=== 한국 주식 ===
{'name': 'SamsungElec', 'current_price': 285500.0, 'prev_close': 268500.0, 'currency': 'KRW'}

=== 오류 케이스 (잘못된 티커) ===


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: INVALID_TICKER_XYZ"}}}


[경고] INVALID_TICKER_XYZ: 가격 데이터를 찾을 수 없습니다.
None


---
## 함수 2 | `get_stock_history()` — 기간별 가격 히스토리

### 요구사항
- 입력: 티커 심볼, 기간 문자열
- 출력: `Open`, `High`, `Low`, `Close`, `Volume` 컬럼이 있는 DataFrame
- 기간 매핑:

```
'1주' → '1wk'
'1달' → '1mo'
'3달' → '3mo'
'1년' → '1y'
```

In [28]:
def get_stock_history(ticker_symbol, period='1달'):
    """
    주식 종목의 기간별 가격 히스토리를 조회합니다.
    
    Args:
        ticker_symbol (str): 티커 심볼
        period (str): 조회 기간 ('1주', '1달', '3달', '1년')
    
    Returns:
        pd.DataFrame: OHLCV 데이터 또는 오류 시 None
    """
    # 기간 매핑 딕셔너리
    # 힌트: 키는 한글('1주', '1달', '3달', '1년'), 값은 yfinance period 문자열
    period_map = {
        '1주': '1wk',
        '1달': '1mo',
        '3달': '3mo',
        '1년': '1y',
    }
    
    # period_map에서 yfinance 기간값 꺼내기 (없으면 '1mo' 기본값)
    # 힌트: period_map.get(period, '1mo')
    yf_period = period_map.get(period,'1mo')
    
    try:
        ticker = yf.Ticker(ticker_symbol)
        
        # history() 메서드로 데이터 조회
        # 힌트: ticker.history(period=___)
        df = ticker.history(period = yf_period)
        
        # 데이터가 비어있는지 확인
        # 힌트: df.empty 속성 사용
        if df.empty:
            print(f'[경고] {ticker_symbol}: 히스토리 데이터가 없습니다.')
            return None
        
        # 필요한 컬럼만 선택해서 반환
        # 힌트: df[['Open', 'High', 'Low', 'Close', 'Volume']]
        return df[['Open', 'High', 'Low', 'Close', 'Volume']]
    
    except Exception as e:
        print(f'[오류] {ticker_symbol} 히스토리 조회 실패: {e}')
        return None
    
print(f"get_stock_history => {get_stock_history('AAPL', period='1달')}")

get_stock_history =>                                  Open        High         Low       Close  \
Date                                                                        
2026-04-09 00:00:00-04:00  259.000000  261.119995  256.070007  260.489990   
2026-04-10 00:00:00-04:00  259.980011  262.190002  259.019989  260.480011   
2026-04-13 00:00:00-04:00  259.730011  260.179993  256.660004  259.200012   
2026-04-14 00:00:00-04:00  259.250000  261.929993  257.190002  258.829987   
2026-04-15 00:00:00-04:00  258.160004  266.559998  257.809998  266.429993   
2026-04-16 00:00:00-04:00  266.799988  267.160004  261.269989  263.399994   
2026-04-17 00:00:00-04:00  266.959991  272.299988  266.720001  270.230011   
2026-04-20 00:00:00-04:00  270.329987  274.279999  270.290009  273.049988   
2026-04-21 00:00:00-04:00  271.500000  272.799988  265.399994  266.170013   
2026-04-22 00:00:00-04:00  267.820007  273.739990  266.869995  273.170013   
2026-04-23 00:00:00-04:00  275.049988  275.769989  271.

In [ ]:
# 함수 테스트
df = get_stock_history('AAPL', '1달')
print(f'행 수: {len(df)}')
print(df.tail(3))

print()
# 여러 기간 테스트
for period in ['1주', '1달', '3달', '1년']:
    df = get_stock_history('TSLA', period)
    print(f'기간={period} → {len(df)}행')

행 수: 22
                                 Open        High         Low       Close  \
Date                                                                        
2026-05-06 00:00:00-04:00  281.920013  288.029999  281.070007  287.510010   
2026-05-07 00:00:00-04:00  289.269989  292.130005  285.779999  287.440002   
2026-05-08 00:00:00-04:00  290.010010  294.760010  290.000000  293.320007   

                             Volume  
Date                                 
2026-05-06 00:00:00-04:00  58336100  
2026-05-07 00:00:00-04:00  45224300  
2026-05-08 00:00:00-04:00  52631200  

df =>431.20001220703125
기간=1주 → 5행
df =>431.20001220703125
기간=1달 → 22행
df =>436.3500061035156
기간=3달 → 63행
df =>498.8299865722656
기간=1년 → 251행


---
## 함수 3 | `calculate_change()` — 등락가 & 등락률 계산

### 요구사항
- 입력: 현재가, 전일 종가
- 출력: `(등락가, 등락률)` 튜플
- 등락률 = `(현재가 - 전일종가) / 전일종가 × 100`
- 소수점 2자리 반올림

In [32]:
def calculate_change(current_price, prev_close):
    """
    전일 대비 등락가와 등락률을 계산합니다.
    
    Args:
        current_price (float): 현재가
        prev_close (float): 전일 종가
    
    Returns:
        tuple: (등락가, 등락률) — 소수점 2자리
    """
    # 0으로 나누기 방지
    if prev_close == 0:
        return 0, 0
    
    # 등락가 계산: 현재가 - 전일종가
    # 힌트: 뺄셈 연산
    change = current_price - prev_close
    
    # 등락률 계산: (등락가 / 전일종가) × 100
    # 힌트: change / prev_close * 100
    change_pct = (change / prev_close) * 100
    
    # round()로 소수점 2자리 반올림
    return round(change, 2), round(change_pct, 2)



print(f"calculate_change => {calculate_change(289.269989, 290.000000)}")

calculate_change => (-0.73, -0.25)


In [38]:
# 함수 테스트
change, change_pct = calculate_change(185.0, 180.0)
print(f'등락가: {change}, 등락률: {change_pct}%')  # 5.0, 2.78%

change2, change_pct2 = calculate_change(175.0, 180.0)
print(f'등락가: {change2}, 등락률: {change_pct2}%')  # -5.0, -2.78%

# 실제 데이터로 테스트
info = get_stock_info('AAPL')
print()
if info:
    c, cp = calculate_change(info['current_price'], info['prev_close'])
    print(f'AAPL 실제 등락가: {c}, 등락률: {cp}%')

등락가: 5.0, 등락률: 2.78%
등락가: -5.0, 등락률: -2.78%
info ===  {'address1': 'One Apple Park Way', 'city': 'Cupertino', 'state': 'CA', 'zip': '95014', 'country': 'United States', 'phone': '(408) 996-1010', 'website': 'https://www.apple.com', 'industry': 'Consumer Electronics', 'industryKey': 'consumer-electronics', 'industryDisp': 'Consumer Electronics', 'sector': 'Technology', 'sectorKey': 'technology', 'sectorDisp': 'Technology', 'longBusinessSummary': 'Apple Inc. designs, manufactures, and markets smartphones, personal computers, tablets, wearables, and accessories worldwide. The company offers iPhone, a line of smartphones; Mac, a line of personal computers; iPad, a line of multi-purpose tablets; and wearables, home, and accessories comprising AirPods, Apple Vision Pro, Apple TV, Apple Watch, Beats products, and HomePod, as well as Apple branded and third-party accessories. It also provides AppleCare support and cloud services; and operates various platforms, including the App Store that all

---
## 함수 4 | `calculate_moving_average()` — 이동평균 계산 (선택)

### 요구사항
- 입력: 가격 히스토리 DataFrame, 이동평균 기간 리스트
- 출력: 이동평균 컬럼이 추가된 DataFrame
- 예: `periods=[5, 20]` → `MA5`, `MA20` 컬럼 추가

In [49]:
def calculate_moving_average(df, periods=[5, 20]):
    """
    이동평균선을 계산해서 DataFrame에 컬럼으로 추가합니다.
    
    Args:
        df (pd.DataFrame): OHLCV 데이터
        periods (list): 이동평균 기간 리스트 (예: [5, 20, 60])
    
    Returns:
        pd.DataFrame: 이동평균 컬럼이 추가된 DataFrame
    """
    # DataFrame 복사 (원본 수정 방지)
    # 힌트: df.copy()
    result = df.copy()
    
    for period in periods:
        # 컬럼 이름: 'MA5', 'MA20' 등 f-string으로 만들기
        col_name = f'MA{period}'
        
        # pandas rolling().mean()으로 이동평균 계산
        # 힌트: result['Close'].rolling(window=period).mean()
        result[col_name] = result['Close'].rolling(window=period).mean()
    
    return result

print('df = >',df)
print('test = >',calculate_moving_average(df))

df = >                                  Open        High         Low       Close  \
Date                                                                        
2026-02-13 00:00:00-05:00  261.768832  261.988631  255.214858  255.544556   
2026-02-17 00:00:00-05:00  257.812465  266.044901  255.304780  263.637115   
2026-02-18 00:00:00-05:00  263.357380  266.574417  262.208444  264.106689   
2026-02-19 00:00:00-05:00  262.358278  264.236553  259.810608  260.340118   
2026-02-20 00:00:00-05:00  258.731635  264.506313  257.922383  264.336456   
...                               ...         ...         ...         ...   
2026-05-06 00:00:00-04:00  281.660510  287.764872  280.811287  287.245361   
2026-05-07 00:00:00-04:00  289.003717  291.861100  285.516939  287.175415   
2026-05-08 00:00:00-04:00  289.743067  294.488695  289.733067  293.050018   
2026-05-11 00:00:00-04:00  291.980011  293.880005  290.230011  292.679993   
2026-05-12 00:00:00-04:00  292.559998  295.269989  292.559998  294.79

In [39]:
# 함수 테스트
df = get_stock_history('AAPL', '3달')
df_with_ma = calculate_moving_average(df, periods=[5, 20])

print('컬럼 목록:', df_with_ma.columns.tolist())
print(df_with_ma[['Close', 'MA5', 'MA20']].tail(10))

컬럼 목록: ['Open', 'High', 'Low', 'Close', 'Volume', 'MA5', 'MA20']
                                Close         MA5        MA20
Date                                                         
2026-04-29 00:00:00-04:00  269.921326  270.346918  264.118658
2026-04-30 00:00:00-04:00  271.100250  269.931305  264.903936
2026-05-01 00:00:00-04:00  279.882141  271.745636  266.113821
2026-05-04 00:00:00-04:00  276.575165  273.587939  267.011494
2026-05-05 00:00:00-04:00  283.918427  276.279462  268.544083
2026-05-06 00:00:00-04:00  287.245361  279.744269  269.973267
2026-05-07 00:00:00-04:00  287.175415  282.959302  271.319527
2026-05-08 00:00:00-04:00  293.050018  285.592877  272.960016
2026-05-11 00:00:00-04:00  292.679993  288.813843  274.645944
2026-05-12 00:00:00-04:00  294.799988  290.990155  276.456357


---
## (선택) 코인 데이터 — pyupbit
주식 대신 코인으로 만들고 싶다면 아래 함수를 사용하세요.

In [40]:
# pyupbit 탐색
try:
    import pyupbit
    
    # 현재가 조회
    price = pyupbit.get_current_price('KRW-BTC')
    print(f'비트코인 현재가: {price:,}원')
    
    # OHLCV 조회 (count: 최근 N일)
    df_coin = pyupbit.get_ohlcv('KRW-BTC', count=30)
    print(df_coin.tail(3))
    
except ImportError:
    print('pyupbit 미설치: !pip install pyupbit 실행 후 재시도')

비트코인 현재가: 119,514,000.0원
                            open         high          low        close  \
2026-05-11 09:00:00  120461000.0  120700000.0  118739000.0  120095000.0   
2026-05-12 09:00:00  120095000.0  120297000.0  118646000.0  119202000.0   
2026-05-13 09:00:00  119202000.0  120339000.0  119112000.0  119514000.0   

                          volume         value  
2026-05-11 09:00:00  1432.186211  1.713445e+11  
2026-05-12 09:00:00  1157.868766  1.384603e+11  
2026-05-13 09:00:00   555.057038  6.655491e+10  


In [48]:
def get_coin_history(ticker='KRW-BTC', days=30):
    """
    코인 OHLCV 데이터를 조회합니다.
    
    Args:
        ticker (str): 업비트 티커 (예: 'KRW-BTC', 'KRW-ETH')
        days (int): 조회 일수
    
    Returns:
        pd.DataFrame: OHLCV 데이터
    """
    try:
        import pyupbit
        # 힌트: pyupbit.get_ohlcv(ticker, count=days)
        df = pyupbit.get_ohlcv(ticker , days)
        
        # 컬럼명을 yfinance와 동일하게 통일
        # pyupbit: open/high/low/close/volume → Open/High/Low/Close/Volume
        df.columns = [col.capitalize() for col in df.columns]
        return df
    except Exception as e:
        print(f'[오류] 코인 데이터 조회 실패: {e}')
        return None
    
    print('test ==== ',df)

---
## Step 1~2 마무리 체크

아래 항목을 모두 확인한 후 다음 노트북으로 이동하세요!

| 체크 | 항목 |
|:---:|------|
| ☐ | `get_stock_info('AAPL')` 실행 시 딕셔너리 반환 |
| ☐ | `get_stock_info('INVALID')` 실행 시 None 반환 |
| ☐ | `get_stock_history('AAPL', '1달')` 실행 시 DataFrame 반환 |
| ☐ | 4가지 기간 모두 정상 작동 |
| ☐ | `calculate_change(185, 180)` → `(5.0, 2.78)` |
| ☐ | `calculate_moving_average()` 실행 시 MA5, MA20 컬럼 생성 |

### 다음 단계
`02_시각화_실습.ipynb` 으로 이동해서 plotly 차트를 만들어보세요!